# STLab02 - EduCommerce test execution (REQ-07, 09, 10, 14, 16)
Nguyễn Khắc Trọng - 23110217. Chạy theo thứ tự: ghi file -> chạy test -> xem report -> phân tích FAIL -> regression.

In [ ]:
%%writefile ecommerce_app.py
"""EduCommerce: in-memory application for practicing checkout and automated testing."""
# NGUỒN: STLab02 Instruction mục 2.6 (thầy Tuấn cung cấp). KHÔNG sửa code ở đây khi chạy test;
# nếu muốn sửa bug để regression, làm trên bản sao ecommerce_app_fixed.py.
from dataclasses import dataclass, field
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional

D = lambda x: Decimal(str(x))
def money(x): return D(x).quantize(D('1'), rounding=ROUND_HALF_UP)

@dataclass
class Product:
    sku: str; title: str; price: Decimal; stock: int; active: bool = True

@dataclass
class Customer:
    customer_id: str; authenticated: bool = True; used_codes: set = field(default_factory=set)

@dataclass
class CartLine:
    sku: str; quantity: int

@dataclass
class Voucher:
    code: str; percent: int; min_subtotal: Decimal; cap: Decimal; active: bool = True

@dataclass
class Order:
    order_id: str; customer_id: str; total: Decimal; status: str; lines: list; discount: Decimal; shipping: Decimal; payment_method: str

class Catalog:
    def __init__(self):
        self.products = {'A': Product('A', 'Wireless Mouse', D(200000), 20),
                         'B': Product('B', 'Mechanical Keyboard', D(300000), 15),
                         'C': Product('C', 'USB-C Hub', D(150000), 12)}
    def get(self, sku):
        p = self.products.get(sku)
        if not p or not p.active: raise ValueError('PRODUCT_UNAVAILABLE')
        return p

class Cart:
    def __init__(self, catalog): self.catalog = catalog; self.lines = []
    def add(self, sku, quantity):
        self.catalog.get(sku)
        if not isinstance(quantity, int) or quantity < 1: raise ValueError('INVALID_QUANTITY')
        existing = next((x for x in self.lines if x.sku == sku), None)
        if existing: existing.quantity += quantity
        else: self.lines.append(CartLine(sku, quantity))
    def subtotal(self): return sum((self.catalog.get(x.sku).price * x.quantity for x in self.lines), D(0))

class Inventory:
    def __init__(self, catalog): self.catalog = catalog
    def reserve(self, lines):
        for line in lines:
            if self.catalog.get(line.sku).stock < line.quantity: raise ValueError('OUT_OF_STOCK')
        for line in lines: self.catalog.get(line.sku).stock -= line.quantity
    def release(self, lines):
        for line in lines: self.catalog.get(line.sku).stock += line.quantity

class Promotions:
    def __init__(self): self.codes = {'SAVE10': Voucher('SAVE10', 10, D(500000), D(100000))}
    def discount(self, code, subtotal, customer):
        if not code: return D(0)
        v = self.codes.get(code)
        if not v or not v.active: raise ValueError('INVALID_VOUCHER')
        if not customer.authenticated: raise ValueError('LOGIN_REQUIRED')
        if code in customer.used_codes: raise ValueError('VOUCHER_USED')
        # Validate minimum subtotal for voucher eligibility.
        if subtotal <= v.min_subtotal: raise ValueError('MIN_SUBTOTAL_NOT_MET')
        return min(money(subtotal * v.percent / D(100)), v.cap)

class Shipping:
    def calculate(self, subtotal, area):
        if area not in ('inner', 'outer'): raise ValueError('INVALID_AREA')
        return D(0) if area == 'inner' and subtotal >= D(500000) else (D(30000) if area == 'inner' else D(50000))

class Payments:
    def __init__(self): self.charges = []
    def collect(self, amount, method, approved, key):
        if method == 'COD': return 'PENDING_COD'
        if method != 'ONLINE': raise ValueError('INVALID_PAYMENT_METHOD')
        if not approved: raise ValueError('PAYMENT_DECLINED')
        self.charges.append({'key': key, 'amount': amount})
        return 'PAID'

class Checkout:
    def __init__(self, catalog=None):
        self.catalog = catalog or Catalog(); self.inventory = Inventory(self.catalog)
        self.promotions = Promotions(); self.shipping = Shipping(); self.payments = Payments()
        self.orders = {}; self.by_key = {}; self.counter = 0
    def place(self, customer, cart, area='inner', voucher=None, method='COD', approved=True, key='REQ-001'):
        if not customer.authenticated: raise ValueError('LOGIN_REQUIRED')
        if not cart.lines: raise ValueError('EMPTY_CART')
        subtotal = cart.subtotal()
        discount = self.promotions.discount(voucher, subtotal, customer)
        shipping = self.shipping.calculate(subtotal, area)
        total = max(D(0), subtotal - discount + shipping)
        # Reserve stock during the checkout transaction.
        self.inventory.reserve(cart.lines)
        if key in self.by_key: return self.orders[self.by_key[key]]
        # Process the online payment outcome.
        payment_state = self.payments.collect(total, method, approved, key)
        self.counter += 1
        order = Order(f'ORD-{self.counter:04d}', customer.customer_id, total, payment_state,
                      [(x.sku, x.quantity) for x in cart.lines], discount, shipping, method)
        self.orders[order.order_id] = order; self.by_key[key] = order.order_id
        if voucher: customer.used_codes.add(voucher)
        return order

def fixture(subtotal_case='600k'):
    service = Checkout(); customer = Customer('U_NEW'); cart = Cart(service.catalog)
    if subtotal_case == '500k': cart.add('A', 1); cart.add('B', 1)
    elif subtotal_case == '499999':
        cart.add('A', 1); cart.add('B', 1); service.catalog.products['B'].price = D(299999)
    elif subtotal_case == '1000k': cart.add('A', 5)
    elif subtotal_case == '1500k': cart.add('B', 5)
    elif subtotal_case == '600k': cart.add('A', 3)
    elif subtotal_case == '300k': cart.add('B', 1)
    else: raise ValueError('UNKNOWN_FIXTURE')
    return service, customer, cart


In [ ]:
%%writefile ecommerce_app_fixed.py
"""EduCommerce: in-memory application for practicing checkout and automated testing."""
# BẢN SỬA (chỉ để regression REQ-16; fix rollback khi payment lỗi). NGUỒN GỐC: STLab02 Instruction mục 2.6 (thầy Tuấn cung cấp). KHÔNG sửa code ở đây khi chạy test;
# nếu muốn sửa bug để regression, làm trên bản sao ecommerce_app_fixed.py.
from dataclasses import dataclass, field
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional

D = lambda x: Decimal(str(x))
def money(x): return D(x).quantize(D('1'), rounding=ROUND_HALF_UP)

@dataclass
class Product:
    sku: str; title: str; price: Decimal; stock: int; active: bool = True

@dataclass
class Customer:
    customer_id: str; authenticated: bool = True; used_codes: set = field(default_factory=set)

@dataclass
class CartLine:
    sku: str; quantity: int

@dataclass
class Voucher:
    code: str; percent: int; min_subtotal: Decimal; cap: Decimal; active: bool = True

@dataclass
class Order:
    order_id: str; customer_id: str; total: Decimal; status: str; lines: list; discount: Decimal; shipping: Decimal; payment_method: str

class Catalog:
    def __init__(self):
        self.products = {'A': Product('A', 'Wireless Mouse', D(200000), 20),
                         'B': Product('B', 'Mechanical Keyboard', D(300000), 15),
                         'C': Product('C', 'USB-C Hub', D(150000), 12)}
    def get(self, sku):
        p = self.products.get(sku)
        if not p or not p.active: raise ValueError('PRODUCT_UNAVAILABLE')
        return p

class Cart:
    def __init__(self, catalog): self.catalog = catalog; self.lines = []
    def add(self, sku, quantity):
        self.catalog.get(sku)
        if not isinstance(quantity, int) or quantity < 1: raise ValueError('INVALID_QUANTITY')
        existing = next((x for x in self.lines if x.sku == sku), None)
        if existing: existing.quantity += quantity
        else: self.lines.append(CartLine(sku, quantity))
    def subtotal(self): return sum((self.catalog.get(x.sku).price * x.quantity for x in self.lines), D(0))

class Inventory:
    def __init__(self, catalog): self.catalog = catalog
    def reserve(self, lines):
        for line in lines:
            if self.catalog.get(line.sku).stock < line.quantity: raise ValueError('OUT_OF_STOCK')
        for line in lines: self.catalog.get(line.sku).stock -= line.quantity
    def release(self, lines):
        for line in lines: self.catalog.get(line.sku).stock += line.quantity

class Promotions:
    def __init__(self): self.codes = {'SAVE10': Voucher('SAVE10', 10, D(500000), D(100000))}
    def discount(self, code, subtotal, customer):
        if not code: return D(0)
        v = self.codes.get(code)
        if not v or not v.active: raise ValueError('INVALID_VOUCHER')
        if not customer.authenticated: raise ValueError('LOGIN_REQUIRED')
        if code in customer.used_codes: raise ValueError('VOUCHER_USED')
        # Validate minimum subtotal for voucher eligibility.
        if subtotal <= v.min_subtotal: raise ValueError('MIN_SUBTOTAL_NOT_MET')
        return min(money(subtotal * v.percent / D(100)), v.cap)

class Shipping:
    def calculate(self, subtotal, area):
        if area not in ('inner', 'outer'): raise ValueError('INVALID_AREA')
        return D(0) if area == 'inner' and subtotal >= D(500000) else (D(30000) if area == 'inner' else D(50000))

class Payments:
    def __init__(self): self.charges = []
    def collect(self, amount, method, approved, key):
        if method == 'COD': return 'PENDING_COD'
        if method != 'ONLINE': raise ValueError('INVALID_PAYMENT_METHOD')
        if not approved: raise ValueError('PAYMENT_DECLINED')
        self.charges.append({'key': key, 'amount': amount})
        return 'PAID'

class Checkout:
    def __init__(self, catalog=None):
        self.catalog = catalog or Catalog(); self.inventory = Inventory(self.catalog)
        self.promotions = Promotions(); self.shipping = Shipping(); self.payments = Payments()
        self.orders = {}; self.by_key = {}; self.counter = 0
    def place(self, customer, cart, area='inner', voucher=None, method='COD', approved=True, key='REQ-001'):
        if not customer.authenticated: raise ValueError('LOGIN_REQUIRED')
        if not cart.lines: raise ValueError('EMPTY_CART')
        subtotal = cart.subtotal()
        discount = self.promotions.discount(voucher, subtotal, customer)
        shipping = self.shipping.calculate(subtotal, area)
        total = max(D(0), subtotal - discount + shipping)
        # Reserve stock during the checkout transaction.
        self.inventory.reserve(cart.lines)
        if key in self.by_key: return self.orders[self.by_key[key]]
        # Process the online payment outcome. FIX: nếu payment lỗi (declined / method sai) thì trả stock đã giữ.
        try:
            payment_state = self.payments.collect(total, method, approved, key)
        except ValueError:
            self.inventory.release(cart.lines)
            raise
        self.counter += 1
        order = Order(f'ORD-{self.counter:04d}', customer.customer_id, total, payment_state,
                      [(x.sku, x.quantity) for x in cart.lines], discount, shipping, method)
        self.orders[order.order_id] = order; self.by_key[key] = order.order_id
        if voucher: customer.used_codes.add(voucher)
        return order

def fixture(subtotal_case='600k'):
    service = Checkout(); customer = Customer('U_NEW'); cart = Cart(service.catalog)
    if subtotal_case == '500k': cart.add('A', 1); cart.add('B', 1)
    elif subtotal_case == '499999':
        cart.add('A', 1); cart.add('B', 1); service.catalog.products['B'].price = D(299999)
    elif subtotal_case == '1000k': cart.add('A', 5)
    elif subtotal_case == '1500k': cart.add('B', 5)
    elif subtotal_case == '600k': cart.add('A', 3)
    elif subtotal_case == '300k': cart.add('B', 1)
    else: raise ValueError('UNKNOWN_FIXTURE')
    return service, customer, cart


In [ ]:
%%writefile test_runner.py
"""STLab02 - CSV-driven test execution for EduCommerce (REQ-07, 09, 10, 14, 16).

Luồng: CSV -> validate schema -> fixture riêng -> dispatcher (op) -> so sánh expected/actual
       -> PASS / FAIL / BLOCKED -> CSV + Excel + HTML dashboard + chart.
ecommerce_app.py là code của thầy, KHÔNG sửa; mọi expected lấy từ acceptance criteria.
"""
import base64, csv, datetime, html, io, json, time
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import importlib, os
_app = importlib.import_module(os.environ.get("APP_MODULE", "ecommerce_app"))  # đổi sang ecommerce_app_fixed để regression
Cart, Checkout, Customer, fixture = _app.Cart, _app.Checkout, _app.Customer, _app.fixture

ROOT = Path(__file__).resolve().parent
REQUIRED_COLUMNS = ["test_case_id", "requirement_id", "condition_id", "scenario_id", "title", "priority",
                    "precondition", "fixture_key", "input_json", "expected_type", "expected_json",
                    "evidence_reference"]


class Blocked(Exception):
    """Test không thể chạy (dữ liệu/fixture hỏng) - khác với FAIL của sản phẩm."""


def make_fixture(key):
    """Mỗi case một môi trường mới: (service, customer, cart)."""
    if key == "empty":
        service = Checkout()
        return service, Customer("U_NEW"), Cart(service.catalog)
    if key == "used_save10":
        service, customer, cart = fixture("600k")
        customer.used_codes.add("SAVE10")
        return service, customer, cart
    try:
        return fixture(key)          # 300k, 500k, 499999, 600k, 1000k, 1500k của thầy
    except ValueError as exc:
        raise Blocked(f"UNKNOWN_FIXTURE:{key}") from exc


def stock_of(app):
    return {k: p.stock for k, p in app.catalog.products.items()}


def run_operation(app, user, cart, spec):
    """Dispatcher: gọi hàm nghiệp vụ theo spec['op']. Trả về (order, error_code)."""
    op = spec.get("op")
    try:
        if op == "ship":
            fee = app.shipping.calculate(cart.subtotal(), spec["area"])
            return {"shipping_fee": f"{fee:.0f}"}, ""
        if op in ("place", "place_twice"):
            kwargs = dict(area=spec.get("area", "inner"), voucher=spec.get("voucher"),
                          method=spec.get("method", "COD"), approved=spec.get("approved", True))
            order = app.place(user, cart, key=spec.get("key", "K1"), **kwargs)
            if op == "place_twice":                       # lần 2: khác key, cùng giỏ
                order = app.place(user, cart, key=spec.get("key", "K1") + "-2", **kwargs)
            return order, ""
        raise Blocked(f"UNKNOWN_OP:{op}")
    except ValueError as exc:                             # lỗi nghiệp vụ -> ghi lại để so với expected
        return None, str(exc)


def observe(app, user, order, error, before):
    """Gom actual: giá trị trả về + STATE sau thao tác (stock, orders, charges, used_codes)."""
    obs = {"error": error or "NONE", "orders": len(app.orders), "charges": len(app.payments.charges),
           "stock_unchanged": stock_of(app) == before, "used_codes": sorted(user.used_codes)}
    if isinstance(order, dict):
        obs.update(order)
    elif order is not None:
        obs.update(status=order.status, shipping=f"{order.shipping:.0f}", discount=f"{order.discount:.0f}",
                   total=f"{order.total:.0f}")
    return obs


def execute_case(row):
    started = time.perf_counter()
    result = {**row, "execution_status": "", "actual_result": "", "error_message": "", "mismatch": ""}
    try:
        try:
            spec, expected = json.loads(row["input_json"]), json.loads(row["expected_json"])
        except json.JSONDecodeError as exc:
            raise Blocked(f"BAD_JSON:{exc.msg}") from exc
        app, user, cart = make_fixture(row["fixture_key"])          # fixture mới cho từng case
        before = stock_of(app)
        order, error = run_operation(app, user, cart, spec)
        actual = observe(app, user, order, error, before)
        mismatches = [f"{k}: expected {v!r} but actual {actual.get(k)!r}"
                      for k, v in expected.items() if actual.get(k) != v]
        result["actual_result"] = json.dumps({k: actual.get(k) for k in expected}, ensure_ascii=False)
        result["error_message"] = actual["error"] if actual["error"] != "NONE" else ""
        result["mismatch"] = "; ".join(mismatches)
        result["execution_status"] = "PASS" if not mismatches else "FAIL"
    except Blocked as exc:
        result.update(execution_status="BLOCKED", error_message=str(exc))
    except Exception as exc:                                         # lỗi bất ngờ của app: FAIL, không nuốt
        result.update(execution_status="FAIL", error_message=f"UNEXPECTED:{type(exc).__name__}:{exc}")
    result["duration_ms"] = round((time.perf_counter() - started) * 1000, 3)
    result["executed_at"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    return result


def load_cases(csv_path):
    with Path(csv_path).open(newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        missing = [c for c in REQUIRED_COLUMNS if c not in (reader.fieldnames or [])]
        if missing:
            raise ValueError(f"MISSING_COLUMNS:{missing}")
        rows = list(reader)
    ids = [r["test_case_id"] for r in rows]
    if len(ids) != len(set(ids)):
        raise ValueError("DUPLICATE_TEST_IDS")
    return rows


def chart_png(df):
    table = df.groupby(["requirement_id", "execution_status"]).size().unstack(fill_value=0)
    for col in ("PASS", "FAIL", "BLOCKED"):
        if col not in table:
            table[col] = 0
    fig, ax = plt.subplots(figsize=(7, 4))
    table[["PASS", "FAIL", "BLOCKED"]].plot(kind="bar", stacked=True, ax=ax,
                                            color=["#16a34a", "#dc2626", "#f59e0b"])
    ax.set_title("EduCommerce - test results by requirement"); ax.set_ylabel("Test cases"); ax.set_xlabel("")
    plt.xticks(rotation=0); fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=130); plt.close(fig)
    return buf.getvalue()


def write_reports(df, out):
    counts = Counter(df.execution_status)
    total = len(df)
    summary = df.groupby(["requirement_id", "execution_status"]).size().unstack(fill_value=0).reset_index()
    totals = pd.DataFrame([{"executed": total, "passed": counts["PASS"], "failed": counts["FAIL"],
                            "blocked": counts["BLOCKED"], "pass_rate_%": round(100 * counts["PASS"] / total, 1),
                            "requirements": df.requirement_id.nunique(), "conditions": df.condition_id.nunique(),
                            "scenarios": df.scenario_id.nunique()}])
    trace = df[["requirement_id", "condition_id", "scenario_id", "test_case_id", "execution_status"]]
    df.to_csv(out / "test_execution.csv", index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(out / "test_report.xlsx", engine="openpyxl") as xls:
        df.to_excel(xls, index=False, sheet_name="Execution")
        summary.to_excel(xls, index=False, sheet_name="By requirement")
        trace.to_excel(xls, index=False, sheet_name="Traceability")
        totals.to_excel(xls, index=False, sheet_name="Summary")
    png = chart_png(df)
    (out / "chart_by_requirement.png").write_bytes(png)
    cols = ["test_case_id", "requirement_id", "scenario_id", "expected_json", "actual_result", "execution_status", "duration_ms"]
    rows = "".join("<tr class='" + str(r["execution_status"]) + "'>" +
                   "".join(f"<td>{html.escape(str(r[c]))}</td>" for c in cols) + "</tr>"
                   for _, r in df.iterrows())
    page = (
        "<!DOCTYPE html><html><head><meta charset='utf-8'><title>EduCommerce STLab02 Report</title><style>"
        "body{font:14px Arial;margin:30px;color:#14233c}h1{color:#17385f}.m{display:inline-block;margin:8px;"
        "padding:12px 18px;border-radius:10px;background:#edf3f9;font-size:20px}.row{display:flex;gap:24px;"
        "flex-wrap:wrap;align-items:flex-start}table{border-collapse:collapse;font-size:12px}"
        "td,th{border:1px solid #cbd5e1;padding:6px;text-align:left}th{background:#16385e;color:#fff}"
        "tr.FAIL td{background:#fee2e2}tr.BLOCKED td{background:#fef3c7}</style></head><body>"
        "<h1>EduCommerce - STLab02 automated test execution</h1>"
        f"<div class='m'>Total {total}</div><div class='m'>PASS {counts['PASS']}</div>"
        f"<div class='m'>FAIL {counts['FAIL']}</div><div class='m'>BLOCKED {counts['BLOCKED']}</div>"
        f"<div class='m'>Pass rate {100 * counts['PASS'] / total:.1f}%</div>"
        "<div class='row'><img src='data:image/png;base64," + base64.b64encode(png).decode() + "' width='520'>"
        "<table><tr>" + "".join(f"<th>{c}</th>" for c in cols) + "</tr>" + rows + "</table></div></body></html>")
    (out / "test_dashboard.html").write_text(page, encoding="utf-8")
    return counts


def run_suite(csv_path=None, out_dir=None):
    csv_path = Path(csv_path or ROOT / "test_cases.csv")
    out = Path(out_dir or ROOT / "reports"); out.mkdir(parents=True, exist_ok=True)
    cases = load_cases(csv_path)
    df = pd.DataFrame([execute_case(c) for c in cases])
    counts = write_reports(df, out)
    print(f"Executed {len(df)} | PASS {counts['PASS']} | FAIL {counts['FAIL']} | BLOCKED {counts['BLOCKED']}")
    pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 70)
    print(df[["test_case_id", "requirement_id", "execution_status", "duration_ms", "mismatch"]].to_string(index=False))
    return df, out


if __name__ == "__main__":
    import sys
    run_suite(out_dir=sys.argv[1] if len(sys.argv) > 1 else None)   # python test_runner.py [thư_mục_report]


In [ ]:
%%writefile test_cases.csv
test_case_id,requirement_id,condition_id,scenario_id,title,priority,precondition,test_steps,fixture_key,input_json,expected_type,expected_json,execution_status,actual_result,error_message,evidence_reference
TC-SHP-001,REQ-09,COND-SHP-01,TS-SHP-01,"Nội thành, subtotal đúng 500.000 -> miễn ship (boundary)",P0,U_NEW đã đăng nhập; giỏ có A x1 + B x1 (subtotal 500.000),"1. make_fixture('500k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Shipping.calculate(cart.subtotal(), 'inner'). 3. So sánh phí ship với expected.",500k,"{""op"": ""ship"", ""area"": ""inner""}",value,"{""shipping_fee"": ""0""}",,,,"E07,E09"
TC-SHP-002,REQ-09,COND-SHP-01,TS-SHP-01,"Nội thành, subtotal 499.999 -> phí 30.000 (boundary)",P0,"U_NEW đã đăng nhập; giỏ có A x1 + B x1, giá B = 299.999 (subtotal 499.999)","1. make_fixture('499999') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Shipping.calculate(cart.subtotal(), 'inner'). 3. So sánh phí ship với expected.",499999,"{""op"": ""ship"", ""area"": ""inner""}",value,"{""shipping_fee"": ""30000""}",,,,E07
TC-SHP-003,REQ-09,COND-SHP-02,TS-SHP-02,"Nội thành, subtotal 600.000 -> miễn ship",P1,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Shipping.calculate(cart.subtotal(), 'inner'). 3. So sánh phí ship với expected.",600k,"{""op"": ""ship"", ""area"": ""inner""}",value,"{""shipping_fee"": ""0""}",,,,E07
TC-SHP-004,REQ-09,COND-SHP-02,TS-SHP-02,"Nội thành, subtotal 300.000 -> phí 30.000",P1,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có B x1 (subtotal 300.000)","1. make_fixture('300k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Shipping.calculate(cart.subtotal(), 'inner'). 3. So sánh phí ship với expected.",300k,"{""op"": ""ship"", ""area"": ""inner""}",value,"{""shipping_fee"": ""30000""}",,,,E07
TC-SHP-005,REQ-09,COND-SHP-03,TS-SHP-03,"Ngoại thành, subtotal 600.000 -> phí 50.000 (không được miễn)",P1,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Shipping.calculate(cart.subtotal(), 'outer'). 3. So sánh phí ship với expected.",600k,"{""op"": ""ship"", ""area"": ""outer""}",value,"{""shipping_fee"": ""50000""}",,,,E07
TC-SHP-006,REQ-09,COND-SHP-03,TS-SHP-03,"Ngoại thành, subtotal 300.000 -> phí 50.000",P1,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có B x1 (subtotal 300.000)","1. make_fixture('300k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Shipping.calculate(cart.subtotal(), 'outer'). 3. So sánh phí ship với expected.",300k,"{""op"": ""ship"", ""area"": ""outer""}",value,"{""shipping_fee"": ""50000""}",,,,E07
TC-PAY-001,REQ-10,COND-PAY-01,TS-PAY-01,COD -> trạng thái PENDING_COD,P0,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='COD', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""method"": ""COD""}",value,"{""error"": ""NONE"", ""status"": ""PENDING_COD"", ""orders"": 1}",,,,E07
TC-PAY-002,REQ-10,COND-PAY-02,TS-PAY-02,ONLINE approved -> PAID,P0,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='ONLINE', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""method"": ""ONLINE"", ""approved"": true}",value,"{""error"": ""NONE"", ""status"": ""PAID"", ""charges"": 1}",,,,E07
TC-PAY-003,REQ-10,COND-PAY-03,TS-PAY-03,"ONLINE declined -> báo lỗi, không tạo đơn (negative)",P0,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='ONLINE', approved=False, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""method"": ""ONLINE"", ""approved"": false}",error,"{""error"": ""PAYMENT_DECLINED"", ""orders"": 0}",,,,E07
TC-PAY-004,REQ-10,COND-PAY-03,TS-PAY-03,ONLINE declined -> không ghi nhận charge nào (negative),P1,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='ONLINE', approved=False, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""method"": ""ONLINE"", ""approved"": false}",state,"{""charges"": 0}",,,,E07
TC-CRT-001,REQ-14,COND-CRT-01,TS-CRT-01,Giỏ rỗng -> EMPTY_CART (negative),P0,U_NEW đã đăng nhập; giỏ hàng rỗng (không có dòng nào),"1. make_fixture('empty') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='COD', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",empty,"{""op"": ""place""}",error,"{""error"": ""EMPTY_CART""}",,,,E07
TC-CRT-002,REQ-14,COND-CRT-01,TS-CRT-01,"Giỏ rỗng -> không tạo đơn, kho không đổi (negative, state)",P0,U_NEW đã đăng nhập; giỏ hàng rỗng (không có dòng nào),"1. make_fixture('empty') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='COD', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",empty,"{""op"": ""place""}",state,"{""orders"": 0, ""stock_unchanged"": true}",,,,E07
TC-CRT-003,REQ-14,COND-CRT-02,TS-CRT-02,Giỏ có 1 sản phẩm -> checkout thành công (positive),P1,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có B x1 (subtotal 300.000)","1. make_fixture('300k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='COD', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",300k,"{""op"": ""place""}",value,"{""error"": ""NONE"", ""orders"": 1}",,,,E07
TC-PMS-001,REQ-16,COND-PMS-01,TS-PMS-01,Phương thức WALLET -> bị từ chối (negative),P0,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='WALLET', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""method"": ""WALLET""}",error,"{""error"": ""INVALID_PAYMENT_METHOD""}",,,,E07
TC-PMS-002,REQ-16,COND-PMS-02,TS-PMS-02,"WALLET bị từ chối -> không tạo đơn, kho không đổi (state)",P0,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='WALLET', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""method"": ""WALLET""}",state,"{""orders"": 0, ""stock_unchanged"": true}",,,,"E07,E09"
TC-PMS-003,REQ-16,COND-PMS-01,TS-PMS-01,Phương thức viết thường 'cod' -> bị từ chối (boundary),P1,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='cod', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""method"": ""cod""}",error,"{""error"": ""INVALID_PAYMENT_METHOD""}",,,,E07
TC-PMS-004,REQ-16,COND-PMS-02,TS-PMS-02,"Phương thức rỗng -> không tạo đơn, kho không đổi (negative, state)",P1,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher=None, method='', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""method"": """"}",state,"{""orders"": 0, ""stock_unchanged"": true}",,,,"E07,E09"
TC-VCH-001,REQ-07,COND-VCH-01,TS-VCH-01,Mã không tồn tại FAKE10 -> INVALID_VOUCHER (negative),P0,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher='FAKE10', method='COD', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""voucher"": ""FAKE10""}",error,"{""error"": ""INVALID_VOUCHER"", ""orders"": 0}",,,,E07
TC-VCH-002,REQ-07,COND-VCH-02,TS-VCH-02,Mã SAVE10 đã dùng -> VOUCHER_USED (negative),P0,"U_NEW đã đăng nhập, đã dùng SAVE10 (used_codes={SAVE10}); giỏ có A x3 (subtotal 600.000)","1. make_fixture('used_save10') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher='SAVE10', method='COD', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",used_save10,"{""op"": ""place"", ""voucher"": ""SAVE10""}",error,"{""error"": ""VOUCHER_USED"", ""orders"": 0}",,,,E07
TC-VCH-003,REQ-07,COND-VCH-03,TS-VCH-03,Thanh toán ONLINE bị từ chối -> mã chưa bị đánh dấu used (state),P0,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher='SAVE10', method='ONLINE', approved=False, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""voucher"": ""SAVE10"", ""method"": ""ONLINE"", ""approved"": false}",state,"{""error"": ""PAYMENT_DECLINED"", ""used_codes"": []}",,,,E07
TC-VCH-004,REQ-07,COND-VCH-03,TS-VCH-03,Đặt hàng thành công với SAVE10 -> mã được đánh dấu used (state),P1,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher='SAVE10', method='COD', approved=True, key='K1'); bắt ValueError nếu có. 3. Quan sát order, mã lỗi và state (orders, charges, kho, used_codes). 4. So sánh với expected_json.",600k,"{""op"": ""place"", ""voucher"": ""SAVE10""}",state,"{""error"": ""NONE"", ""used_codes"": [""SAVE10""]}",,,,E07
TC-VCH-005,REQ-07,COND-VCH-02,TS-VCH-02,Đặt lần 2 với SAVE10 sau khi lần 1 thành công -> VOUCHER_USED (negative),P1,"U_NEW đã đăng nhập, chưa dùng mã; giỏ có A x3 (subtotal 600.000)","1. make_fixture('600k') tạo Checkout, Customer, Cart mới (kho A=20, B=15, C=12). 2. Gọi Checkout.place(user, cart, area='inner', voucher='SAVE10', method='COD', approved=True, key='K1'). 3. Gọi lần 2 với key='K1-2', cùng giỏ. 4. Quan sát mã lỗi và state. 5. So sánh với expected_json.",600k,"{""op"": ""place_twice"", ""voucher"": ""SAVE10""}",error,"{""error"": ""VOUCHER_USED"", ""orders"": 1}",,,,E07


In [ ]:
!ls -lh *.py *.csv
!date -u
!python --version

In [ ]:
import inspect, test_runner as tr
print(inspect.getsource(tr.make_fixture))

In [ ]:
print(inspect.getsource(tr.load_cases))

In [ ]:
print(inspect.getsource(tr.run_operation))

In [ ]:
print(inspect.getsource(tr.observe))

In [ ]:
print(inspect.getsource(tr.execute_case))

In [ ]:
!python test_runner.py reports

In [ ]:
import pandas as pd
df = pd.read_csv('reports/test_execution.csv')
display(df.groupby(['requirement_id','execution_status']).size().unstack(fill_value=0))
df[['test_case_id','execution_status','duration_ms','executed_at']].head(6)

In [ ]:
from IPython.display import HTML
HTML(open('reports/test_dashboard.html').read())

In [ ]:
df[df.execution_status=='FAIL'][['test_case_id','expected_json','actual_result','mismatch']]

In [ ]:
import ecommerce_app
print(inspect.getsource(ecommerce_app.Checkout.place))

In [ ]:
from ecommerce_app import fixture
app, user, cart = fixture('600k')
before = {k: p.stock for k, p in app.catalog.products.items()}
try:
    app.place(user, cart, method='WALLET')
except ValueError as e:
    print('Lỗi:', e)
print('Kho trước:', before)
print('Kho sau :', {k: p.stock for k, p in app.catalog.products.items()}, '| số đơn:', len(app.orders))

In [ ]:
!APP_MODULE=ecommerce_app_fixed python test_runner.py reports_after_fix | head -1